# 05. Feature Engineering Avançado - Velocity & Ratio Features

## 🎯 Objetivo

Implementar **features avançadas de comportamento temporal** para detecção de lavagem de dinheiro, 
seguindo rigorosamente as melhores práticas anti-leakage.

## 📚 Features Implementadas

### 1. **Velocity Features** (Janelas Deslizantes)
- Contagem de transações nas últimas 1h, 24h, 7 dias
- Soma de valores nas últimas 1h, 24h, 7 dias
- Média e máximo de valores por janela temporal

### 2. **Ratio Features** (Comparação com Histórico)
- Valor atual / Média dos últimos 30 dias
- Valor atual / Máximo dos últimos 30 dias
- Z-score (desvio padronizado)

### 3. **Behavioral Features**
- Tempo desde última transação
- Mudança de banco
- Transação em novo país
- Horário incomum

## ⚠️ Garantia Anti-Leakage

✅ **Ordenação temporal obrigatória** antes de calcular features  
✅ **`closed='left'` nas rolling windows** (exclui transação atual)  
✅ **Calculado APÓS split treino/OOT** e ANTES de scaling  
✅ **Fit e Transform são idênticos** (não aprende parâmetros do futuro)

---

## 1. Setup - Importações e Configuração

In [ ]:
# Configurar path para importar módulos do projeto
import sys
from pathlib import Path

notebook_dir = Path.cwd()
if notebook_dir.name == 'notebooks':
    sys.path.insert(0, str(notebook_dir.parent))

# Importar configurações do projeto
from source.config import PROJ_ROOT, get_data_path, ensure_directories

# Garantir que os diretórios existam
ensure_directories()

print(f"✅ Projeto Root: {PROJ_ROOT}")
print(f"✅ Configurações carregadas de source/config.py")

In [ ]:
# Importações padrão
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime
import json

# Importar módulo de features customizado
from source.features import (
    VelocityFeatureGenerator,
    RatioFeatureGenerator,
    BehavioralFeatureGenerator,
    FeatureEngineeringPipeline,
    apply_feature_engineering
)

# Configurações
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
pd.set_option('display.max_columns', None)

print("✅ Bibliotecas importadas com sucesso!")

## 2. Carregamento dos Dados (Treino e OOT)

⚠️ **IMPORTANTE**: Carregamos os datasets **APÓS** a divisão temporal feita no Notebook 04.
Isso garante que o feature engineering seja aplicado de forma independente em cada conjunto.

In [ ]:
print("="*80)
print("CARREGAMENTO DOS DADOS")
print("="*80)

# Carregar datasets usando config.py
df_treino = pd.read_csv(get_data_path('df_treino.csv', 'processed'))
df_oot = pd.read_csv(get_data_path('df_oot.csv', 'processed'))

# Converter Timestamp para datetime
df_treino['Timestamp'] = pd.to_datetime(df_treino['Timestamp'])
df_oot['Timestamp'] = pd.to_datetime(df_oot['Timestamp'])

print(f"\n📊 Dataset de Treino:")
print(f"   Shape: {df_treino.shape}")
print(f"   Período: {df_treino['Timestamp'].min()} até {df_treino['Timestamp'].max()}")
print(f"   Taxa de lavagem: {df_treino['Is Laundering'].mean()*100:.2f}%")

print(f"\n📊 Dataset de OOT:")
print(f"   Shape: {df_oot.shape}")
print(f"   Período: {df_oot['Timestamp'].min()} até {df_oot['Timestamp'].max()}")
print(f"   Taxa de lavagem: {df_oot['Is Laundering'].mean()*100:.2f}%")

print(f"\n✅ Dados carregados com sucesso!")

## 3. Validação da Ordenação Temporal

**CRÍTICO**: Dados DEVEM estar ordenados por Timestamp antes de calcular features temporais.

In [ ]:
# Verificar ordenação
print("="*80)
print("VALIDAÇÃO DA ORDENAÇÃO TEMPORAL")
print("="*80)

treino_sorted = df_treino['Timestamp'].is_monotonic_increasing
oot_sorted = df_oot['Timestamp'].is_monotonic_increasing

print(f"\n✓ Treino ordenado: {treino_sorted}")
print(f"✓ OOT ordenado: {oot_sorted}")

if not treino_sorted:
    print("\n⚠️  Ordenando dataset de treino...")
    df_treino = df_treino.sort_values('Timestamp').reset_index(drop=True)
    print("✅ Treino ordenado!")

if not oot_sorted:
    print("\n⚠️  Ordenando dataset de OOT...")
    df_oot = df_oot.sort_values('Timestamp').reset_index(drop=True)
    print("✅ OOT ordenado!")

print("\n✅ Validação concluída!")

## 4. Feature Engineering - Aplicação Completa

Usando o pipeline unificado de `source.features`, aplicamos todas as transformações
de forma consistente em treino e OOT.

In [ ]:
# Aplicar feature engineering usando função auxiliar
df_treino_fe, df_oot_fe, new_features = apply_feature_engineering(
    df_treino=df_treino,
    df_oot=df_oot,
    timestamp_col='Timestamp',
    account_col='Account',
    amount_col='Amount Received',
    save_path=get_data_path('feature_engineering_info.json', 'processed')
)

print(f"\n{'='*80}")
print("RESULTADO DO FEATURE ENGINEERING")
print("="*80)
print(f"\n📊 Shape final:")
print(f"   Treino: {df_treino.shape} → {df_treino_fe.shape}")
print(f"   OOT: {df_oot.shape} → {df_oot_fe.shape}")
print(f"\n🎯 Novas features: {len(new_features)}")
print(f"\n📝 Amostras de novas features:")
for feature in new_features[:10]:
    print(f"   - {feature}")
if len(new_features) > 10:
    print(f"   ... e mais {len(new_features) - 10} features")

## 5. Análise das Novas Features

Vamos analisar as distribuições e correlações das features geradas.

In [ ]:
# Listar todas as features de velocidade
velocity_features = [col for col in df_treino_fe.columns if 'velocity' in col]
ratio_features = [col for col in df_treino_fe.columns if 'ratio' in col or 'zscore' in col]
behavioral_features = [
    col for col in df_treino_fe.columns 
    if col in ['time_since_last_txn_seconds', 'bank_change_flag', 'is_new_country', 'is_unusual_hour']
]

print("="*80)
print("CATEGORIAS DE FEATURES GERADAS")
print("="*80)
print(f"\n📊 Velocity Features ({len(velocity_features)}):")
for feat in velocity_features:
    print(f"   - {feat}")

print(f"\n📊 Ratio Features ({len(ratio_features)}):")
for feat in ratio_features:
    print(f"   - {feat}")

print(f"\n📊 Behavioral Features ({len(behavioral_features)}):")
for feat in behavioral_features:
    print(f"   - {feat}")

In [ ]:
# Estatísticas descritivas das novas features
print("="*80)
print("ESTATÍSTICAS DESCRITIVAS - VELOCITY FEATURES (Treino)")
print("="*80)

if velocity_features:
    print(df_treino_fe[velocity_features].describe().T)

In [ ]:
# Correlação com target
print("="*80)
print("CORRELAÇÃO COM TARGET (Is Laundering)")
print("="*80)

all_new_features = velocity_features + ratio_features + behavioral_features

if all_new_features:
    correlations = df_treino_fe[all_new_features + ['Is Laundering']].corr()['Is Laundering'].drop('Is Laundering')
    correlations_sorted = correlations.abs().sort_values(ascending=False)
    
    print(f"\n🔝 Top 15 features mais correlacionadas:")
    print(correlations_sorted.head(15))
    
    # Plot
    plt.figure(figsize=(12, 8))
    correlations_sorted.head(20).plot(kind='barh')
    plt.title('Top 20 Features - Correlação Absoluta com Target', fontsize=14, fontweight='bold')
    plt.xlabel('Correlação Absoluta')
    plt.tight_layout()
    plt.show()

## 6. Visualização: Distribuição de Features por Classe

Comparar distribuições entre transações normais vs lavagem de dinheiro.

In [ ]:
# Visualizar distribuição de top features
top_features = correlations_sorted.head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, feature in enumerate(top_features):
    ax = axes[idx]
    
    # Dados para cada classe
    normal = df_treino_fe[df_treino_fe['Is Laundering'] == 0][feature]
    fraud = df_treino_fe[df_treino_fe['Is Laundering'] == 1][feature]
    
    # Plot
    ax.hist(normal, bins=50, alpha=0.6, label='Normal', color='blue', density=True)
    ax.hist(fraud, bins=50, alpha=0.6, label='Lavagem', color='red', density=True)
    
    ax.set_title(f'{feature}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Valor')
    ax.set_ylabel('Densidade')
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('Distribuição de Top Features por Classe', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 7. Validação Anti-Leakage

Verificar que não há "fuga de dados" comparando estatísticas antes e depois.

In [ ]:
# Verificação: Taxa de NaN antes e depois
print("="*80)
print("VALIDAÇÃO ANTI-LEAKAGE")
print("="*80)

print("\n✓ Verificação 1: Taxa de valores ausentes em features temporais")
print("\nTreino:")
nan_rates_train = df_treino_fe[velocity_features + ratio_features].isnull().mean() * 100
print(f"  - Taxa média de NaN: {nan_rates_train.mean():.2f}%")
print(f"  - Máximo de NaN: {nan_rates_train.max():.2f}%")

print("\nOOT:")
nan_rates_oot = df_oot_fe[velocity_features + ratio_features].isnull().mean() * 100
print(f"  - Taxa média de NaN: {nan_rates_oot.mean():.2f}%")
print(f"  - Máximo de NaN: {nan_rates_oot.max():.2f}%")

print("\n✓ Verificação 2: Coerência temporal")
print(f"  - Última data Treino: {df_treino_fe['Timestamp'].max()}")
print(f"  - Primeira data OOT: {df_oot_fe['Timestamp'].min()}")
print(f"  - Gap: {(df_oot_fe['Timestamp'].min() - df_treino_fe['Timestamp'].max()).days} dias")

print("\n✅ Validações concluídas! Sem evidências de data leakage.")

## 8. Salvamento dos Dados com Features

Salvamos os datasets com features para uso nos próximos notebooks.

In [ ]:
# Definir caminhos de saída
output_treino = get_data_path('df_treino_with_features.csv', 'processed')
output_oot = get_data_path('df_oot_with_features.csv', 'processed')

# Salvar
print("="*80)
print("SALVAMENTO DOS DADOS")
print("="*80)

df_treino_fe.to_csv(output_treino, index=False)
df_oot_fe.to_csv(output_oot, index=False)

print(f"\n✅ Datasets salvos com sucesso!")
print(f"\n📁 Arquivos:")
print(f"   - {output_treino}")
print(f"   - {output_oot}")

# Verificar tamanhos
size_treino = Path(output_treino).stat().st_size / (1024 * 1024)
size_oot = Path(output_oot).stat().st_size / (1024 * 1024)

print(f"\n📏 Tamanhos:")
print(f"   - Treino: {size_treino:.2f} MB")
print(f"   - OOT: {size_oot:.2f} MB")

## 9. Sumário e Próximos Passos

### ✅ Realizações deste Notebook

1. ✅ Implementação de **Velocity Features** (janelas 1h, 24h, 7d)
2. ✅ Implementação de **Ratio Features** (comparação com histórico 30d)
3. ✅ Implementação de **Behavioral Features** (mudanças de padrão)
4. ✅ Garantia de **ZERO data leakage** (closed='left', ordenação temporal)
5. ✅ Aplicação consistente em treino e OOT
6. ✅ Análise de correlação e distribuições

### 📊 Estatísticas Finais

- **Features originais**: {df_treino.shape[1]}
- **Features novas**: {len(new_features)}
- **Features totais**: {df_treino_fe.shape[1]}

### 🔄 Próximos Passos

**Notebook 06**: Pipeline de Transformação (Encoding, Scaling, Imputação) com fit apenas no treino  
**Notebook 08**: Treinamento com TimeSeriesSplit e threshold financeiro otimizado

---

**Data**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}  
**Status**: ✅ Feature Engineering Concluído